# 商品购物篮分析

本教程使用更大的商品订单数据集，分别从商品级和品类级挖掘关联规则。


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print


RANDOM_STATE = 42


def find_project_root(start=None):
    """Find the project root by walking upward to the data directory.

    Args:
        start: Optional start path. Defaults to the current working directory.

    Returns:
        Project root path containing the `data` directory.

    Raises:
        FileNotFoundError: If no parent directory contains `data`.
    """
    current = Path.cwd() if start is None else Path(start).resolve()
    for path in [current, *current.parents]:
        if (path / "data").exists():
            return path
    raise FileNotFoundError("未找到包含 data/ 的项目根目录")


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"

plt.rcParams["font.sans-serif"] = ["SimHei", "Arial Unicode MS", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False


try:
    from mlxtend.frequent_patterns import apriori, association_rules, fpgrowth
except ImportError as exc:
    raise ImportError("请先安装依赖: pip install mlxtend") from exc


## 1. 读取订单并构建购物篮


In [ ]:
orders = pd.read_csv(DATA_DIR / "goods_order.csv", encoding="gbk")
goods_types = pd.read_csv(DATA_DIR / "goods_types.csv", encoding="gbk")

basket = pd.crosstab(orders["id"], orders["Goods"]).astype(bool)
hot_goods = basket.sum().sort_values(ascending=False)
top_goods = hot_goods.head(10)

display(orders.head())
display(basket.head().astype(int))
display(top_goods.to_frame("order_count"))


## 2. 热销商品和品类


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=120)
top_goods.sort_values().plot(kind="barh", ax=axes[0], color="#4C78A8")
axes[0].set_title("热销前十商品")
axes[0].set_xlabel("订单数")

type_counts = (
    hot_goods.rename("order_count")
    .to_frame()
    .join(goods_types.set_index("Goods"), how="left")
    .groupby("Types", dropna=False)["order_count"]
    .sum()
    .sort_values(ascending=False)
)
type_counts.head(10).sort_values().plot(kind="barh", ax=axes[1], color="#F58518")
axes[1].set_title("热销前十品类")
axes[1].set_xlabel("订单数")
plt.tight_layout()
plt.show()

display(type_counts.head(10).to_frame())


## 3. 商品级频繁项集和关联规则


In [ ]:
def association_rules_compat(frequent_itemsets, min_confidence=0.2):
    """Create association rules across supported mlxtend versions.

    Args:
        frequent_itemsets: Data frame returned by Apriori or FP-Growth.
        min_confidence: Minimum confidence threshold for generated rules.

    Returns:
        Association rules data frame returned by mlxtend.
    """
    try:
        return association_rules(
            frequent_itemsets, metric="confidence", min_threshold=min_confidence
        )
    except TypeError:
        return association_rules(
            frequent_itemsets,
            num_itemsets=len(frequent_itemsets),
            metric="confidence",
            min_threshold=min_confidence,
        )


def format_rules(rules):
    """Format association rules for compact display.

    Args:
        rules: Raw rules data frame returned by mlxtend.

    Returns:
        Data frame with readable antecedent and consequent columns.
    """
    selected = rules[
        ["antecedents", "consequents", "support", "confidence", "lift"]
    ].copy()
    selected["antecedents"] = selected["antecedents"].apply(lambda x: ", ".join(sorted(x)))
    selected["consequents"] = selected["consequents"].apply(lambda x: ", ".join(sorted(x)))
    return selected.sort_values(["confidence", "lift"], ascending=False)


frequent_apriori = apriori(basket, min_support=0.055, use_colnames=True)
rules_apriori = association_rules_compat(frequent_apriori, min_confidence=0.2)

frequent_fp = fpgrowth(basket, min_support=0.055, use_colnames=True)
rules_fp = association_rules_compat(frequent_fp, min_confidence=0.2)

display(frequent_apriori.sort_values("support", ascending=False).head(10))
display(format_rules(rules_apriori).head(10).round(4))
display(format_rules(rules_fp).head(10).round(4))


## 4. 品类级关联规则


In [ ]:
orders_with_types = orders.merge(goods_types, on="Goods", how="left")
type_basket = pd.crosstab(orders_with_types["id"], orders_with_types["Types"]).astype(bool)

type_itemsets = apriori(type_basket, min_support=0.25, use_colnames=True)
type_rules = association_rules_compat(type_itemsets, min_confidence=0.2)

display(type_basket.head().astype(int))
display(type_itemsets.sort_values("support", ascending=False))
display(format_rules(type_rules).head(10).round(4))
